#### Objectifs

1. Reprendre automatiquement les 16 événements du notebook 09.
2. Sonifier la pluie et l'oxygène pour chaque événement, avec les mêmes paramètres.
3. Écouter les résultats et préparer les stimuli de ton expérience.

#### Sonification des 16 événements

#### Objectif

Dans ce notebook, je produis les sonifications des 16 extraits temporels sélectionnés et vérifiés dans le notebook 09.

Chaque extrait correspond à deux heures de mesures de précipitations et d'oxygène dissous, compressées en 60 secondes.

Je conserve pour le moment les paramètres sonores utilisés dans le notebook 08. Le choix définitif des sons sera effectué dans une étape ultérieure.

In [ ]:

import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import Audio, display

from strauss.sonification import Sonification
from strauss.sources import Objects
from strauss.score import Score
from strauss.generator import Synthesizer


# Dossier des données
dossier = Path("../data/totalité_juillet_septembre_2025")

# Charger les précipitations
df = pd.read_csv(
    dossier / "pluviometres_16_fusionnes.csv",
    parse_dates=["date"]
)

# Ajouter les mesures d'oxygène
stations_oxygene = ["Q01", "Q02", "Q03", "Q04"]

for station in stations_oxygene:

    oxygene = pd.read_csv(
        dossier / f"{station}.csv",
        sep="\t",
        encoding="cp1252",
        decimal=","
    )

    oxygene = oxygene.iloc[:, :2].copy()
    oxygene.columns = ["date", station]

    oxygene["date"] = pd.to_datetime(
        oxygene["date"].astype(str),
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce"
    )

    oxygene[station] = pd.to_numeric(
        oxygene[station],
        errors="coerce"
    )

    df = df.merge(oxygene, on="date", how="left")

# Garder la période étudiée
df = df[
    (df["date"] >= "2025-07-01") &
    (df["date"] < "2025-10-01")
].sort_values("date").reset_index(drop=True)

# Identifier les pluviomètres
stations_pluie = [
    c for c in df.columns
    if c not in ["date"] + stations_oxygene
]

print("Nombre de pluviomètres :", len(stations_pluie))
print("Nombre de stations d'oxygène :", len(stations_oxygene))
print("Nombre de mesures :", len(df))

assert len(stations_pluie) == 16


#### Sélection des événements

Je reprends les 16 fenêtres de deux heures validées dans le notebook 09.

Huit extraits contiennent un épisode de faible oxygénation et huit autres servent de témoins.

Je conserve exactement les mêmes dates pour éviter de modifier la sélection expérimentale.

In [ ]:
# Charger les événements validés dans le notebook 09
selection = pd.read_csv(
    "../data/selection_stimuli.csv",
    parse_dates=["debut"]
)

# Vérifier la sélection
assert len(selection) == 16
assert (selection["categorie"] == "Avec baisse").sum() == 8
assert (selection["categorie"] == "Sans baisse").sum() == 8
assert selection["debut"].is_unique

extraits = list(
    selection[["categorie", "debut"]].itertuples(
        index=False, name=None
    )
)

display(selection)

#### Paramètres de sonification

Je fixe une durée de 60 secondes pour chaque extrait de deux heures.

Je conserve les mêmes références de normalisation et les mêmes gains pour tous les événements, afin que les différences sonores correspondent aux variations des données.

Pour distinguer les deux phénomènes, je conserve un son synthétique tonal pour l'oxygène et j'utilise provisoirement un bruit de pluie pour les précipitations.

Les quatre stations d'oxygène restent réparties dans l'espace stéréo. Pour la pluie, j'utilise provisoirement une représentation globale centrée, sans attribuer de position arbitraire aux 16 pluviomètres.

In [ ]:
# Durée de chaque sonification
duree = 60

# Références communes à tous les événements
reference_oxygene = 10.0
reference_pluie = 10.0

# Positions stéréo provisoires
positions_oxygene = {
    "Q01": 0.25,
    "Q02": 0.42,
    "Q03": 0.58,
    "Q04": 0.75
}

positions_pluie = np.linspace(
    0.1, 0.9, len(stations_pluie)
)

# Volumes provisoires
volume_oxygene = 0.35
volume_pluie = 0.25


#### Fonction de sonification

Je crée une fonction pour appliquer les mêmes paramètres à toutes les stations.

La hauteur du son varie en fonction de la valeur mesurée. Pour la pluie, le volume est nul lorsque la précipitation est nulle.

Cette approche reprend les essais du notebook 08. Elle reste provisoire, notamment parce que la pluie n'est pas encore représentée par de véritables sons percussifs.

Fonction Strauss:

In [ ]:
def sonifier_station(valeurs, temps, volume, position):

    # Préparer les paramètres sonores
    data_son = {
        "pitch": 1.0,
        "time_evo": temps,
        "pitch_shift": valeurs,
        "volume": volume,
        "azimuth": float(position),
        "polar": 0.5
    }

    sources = Objects(data_son.keys())
    sources.fromdict(data_son)
    sources.apply_mapping_functions()

    generator = Synthesizer()
    generator.load_preset("pitch_mapper")

    score = Score([["A2"]], duree)

    soni = Sonification(
        score, sources, generator, "stereo"
    )

    soni.render()

    # Récupérer les deux canaux audio
    streams = list(soni.out_channels.values())

    audio = np.stack(
        [np.asarray(s.values, dtype=float) for s in streams],
        axis=0
    )

    frequence = int(streams[0].samprate)

    return audio, frequence


#### Génération des 16 sonifications

Pour chaque événement, je récupère les 24 mesures correspondant aux deux heures sélectionnées.

Je sonifie séparément les quatre stations d'oxygène et les 16 pluviomètres, puis je rassemble les pistes.

Je conserve trois résultats pour chaque événement : l'oxygène seul, la pluie seule et le mélange des deux. Cela me permettra de comparer les possibilités sonores avant de choisir la version définitive.

In [ ]:
# Stocker les résultats sans encore enregistrer de WAV
sons_evenements = {}

for numero, (categorie, date) in enumerate(extraits, start=1):

    debut = pd.Timestamp(date)
    fin = debut + pd.Timedelta(hours=2)

    extrait = df[
        (df["date"] >= debut) &
        (df["date"] < fin)
    ].sort_values("date").copy()

    # Vérifier les 24 mesures attendues
    if len(extrait) != 24:
        raise ValueError(
            f"Événement {numero} : {len(extrait)} mesures"
        )

    temps = np.linspace(0, 1, len(extrait))

    # Sonifier l'oxygène
    pistes_oxygene = []

    for station in stations_oxygene:

        valeurs = pd.to_numeric(
            extrait[station], errors="coerce"
        ).to_numpy(dtype=float)

        valeurs = np.clip(
            valeurs / reference_oxygene, 0, 1
        )

        # Silence sur les mesures manquantes
        volume = np.where(
            np.isfinite(valeurs), volume_oxygene, 0.0
        )

        valeurs = np.nan_to_num(valeurs)

        audio, freq = sonifier_station(
            valeurs,
            temps,
            volume,
            positions_oxygene[station]
        )

        pistes_oxygene.append(audio)

    mix_oxygene = np.mean(pistes_oxygene, axis=0)

    # Sonifier la pluie
    pistes_pluie = []

    for i, station in enumerate(stations_pluie):

        valeurs = pd.to_numeric(
            extrait[station], errors="coerce"
        ).to_numpy(dtype=float)

        valeurs = np.clip(
            valeurs / reference_pluie, 0, 1
        )

        # Silence lorsqu'il ne pleut pas
        volume = np.where(
            valeurs > 0, volume_pluie, 0.0
        )

        valeurs = np.nan_to_num(valeurs)

        audio, freq_pluie = sonifier_station(
            valeurs,
            temps,
            volume,
            positions_pluie[i]
        )

        if freq_pluie != freq:
            raise ValueError("Fréquences audio différentes")

        pistes_pluie.append(audio)

    mix_pluie = np.mean(pistes_pluie, axis=0)

    # Mélanger les deux phénomènes
    mix_final = (
        0.5 * mix_oxygene
        + 0.5 * mix_pluie
    )

    # Garder les résultats en mémoire
    sons_evenements[numero] = {
        "date": debut,
        "categorie": categorie,
        "oxygene": mix_oxygene,
        "pluie": mix_pluie,
        "ensemble": mix_final,
        "frequence": freq
    }

    print(
        f"Événement {numero}/16 terminé :",
        debut,
        "-", categorie
    )

print("\nLes 16 événements sont sonifiés.")


#### Écoute des résultats

Je peux maintenant écouter chaque événement et comparer les trois versions : oxygène seul, pluie seule et les deux ensemble.

Cette écoute me permet de repérer les problèmes de volume ou de perception avant de choisir les sons définitifs.

In [ ]:
# Choisir un événement entre 1 et 16
numero = 1

evenement = sons_evenements[numero]

print("Événement :", numero)
print("Date :", evenement["date"])
print("Catégorie :", evenement["categorie"])

print("\nOxygène seul")
display(Audio(
    evenement["oxygene"],
    rate=evenement["frequence"],
    normalize=True
))

print("\nPluie seule")
display(Audio(
    evenement["pluie"],
    rate=evenement["frequence"],
    normalize=True
))

print("\nPluie et oxygène")
display(Audio(
    evenement["ensemble"],
    rate=evenement["frequence"],
    normalize=True
))


#### Bilan

J'ai appliqué les mêmes paramètres de sonification aux 16 extraits sélectionnés.

Pour chaque événement, j'ai produit une version avec l'oxygène, une version avec la pluie et une version combinant les deux phénomènes.

Les sons sont encore provisoires. Avant l'expérience, je devrai confirmer la spatialisation des stations, choisir les timbres définitifs et vérifier les niveaux sonores.

Cette étape permet de disposer d'une première base pour comparer les différentes possibilités de sonification.